# UPDATE APO DARKSTORE


- Membaca data dari CSV
- Mengurutkan berdasarkan `7. Selesai` dari kecil ke besar
- Memberi warna merah → kuning → hijau pada kolom `7. Selesai`
- hasil sebagai **Bitmap `.bmp`**


**IMPORT FILE**

In [ ]:
# Import library
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
import glob

# Cari file CSV di folder /content
file_csv = glob.glob("/content/GLI - NEW SAPA  Performance_ALFAGIFT DARK STORE_Tabel pivot.csv")

print("File CSV yang ditemukan:")
for f in file_csv:
    print(f)

# Pilih file CSV pertama
nama_file = file_csv[0]

print("\nFile yang digunakan:", nama_file)

File CSV yang ditemukan:
/content/GLI - NEW SAPA  Performance_ALFAGIFT DARK STORE_Tabel pivot.csv

File yang digunakan: /content/GLI - NEW SAPA  Performance_ALFAGIFT DARK STORE_Tabel pivot.csv


**BACA DATA**

In [ ]:
# Baca CSV
data = pd.read_csv(nama_file)

# Rapikan nama kolom
data.columns = data.columns.str.strip()

print("Kolom:", data.columns.tolist())
data.head()

Kolom: ['store', 'sort status apo', 'Jumlah Order']


,store,sort status apo,Jumlah Order
0,R899 (DS UJUNG PANDANG)-(RZ01),7. Selesai,154
1,UF29 (DS TENGGILIS SBY)-(UZ01),7. Selesai,123
2,R876 (DS HASANUDDIN GOWA)-(RZ01),7. Selesai,117
3,R860 (DS ABDESIR MKS)-(RZ01),7. Selesai,106
4,BJ15 (DS CIHAMPELAS)-(BZ01),7. Selesai,98


In [ ]:
# Sesuaikan nama kolom dari CSV
# Jika CSV menggunakan nama: store, sort status apo, Jumlah Order

data = data.rename(columns={
    "store": "Nama Toko",
    "sort status apo": "Status"
})

# Pastikan Jumlah Order menjadi angka
data["Jumlah Order"] = pd.to_numeric(
    data["Jumlah Order"],
    errors="coerce"
).fillna(0)

# Rapikan teks
data["Nama Toko"] = data["Nama Toko"].astype(str).str.strip()
data["Status"] = data["Status"].astype(str).str.strip()

data.head()

,Nama Toko,Status,Jumlah Order
0,R899 (DS UJUNG PANDANG)-(RZ01),7. Selesai,154
1,UF29 (DS TENGGILIS SBY)-(UZ01),7. Selesai,123
2,R876 (DS HASANUDDIN GOWA)-(RZ01),7. Selesai,117
3,R860 (DS ABDESIR MKS)-(RZ01),7. Selesai,106
4,BJ15 (DS CIHAMPELAS)-(BZ01),7. Selesai,98


**INPUT TANGGAL DAN JAM**

In [ ]:
# Input tanggal dan jam
tanggal = input("Masukkan tanggal (DD/MM/YYYY): ")
jam = input("Masukkan jam (HH.MM): ")

# Urutan status
urutan_status = [
    "1. New",
    "2. Packing",
    "3. Pesanan Siap",
    "4. Siap Kirim",
    "5. Dalam Pengiriman",
    "6. Tunda",
    "7. Selesai",
    "8. Batal"
]

# Buat pivot table
tabel = pd.pivot_table(
    data,
    index="Nama Toko",
    columns="Status",
    values="Jumlah Order",
    aggfunc="sum",
    fill_value=0
)

# Pastikan semua kolom status tersedia dan urutannya benar
tabel = tabel.reindex(columns=urutan_status, fill_value=0)

# Grand Total setiap toko
tabel["Grand Total"] = tabel.sum(axis=1)

# Urutkan berdasarkan 7. Selesai dari kecil ke besar
tabel = tabel.sort_values(
    by="7. Selesai",
    ascending=True
)

# Jadikan Nama Toko sebagai kolom biasa
tabel = tabel.reset_index()

# Tambahkan No
tabel.insert(0, "No", range(1, len(tabel) + 1))

# Buat baris Grand Total
baris_grand_total = {"No": "", "Nama Toko": "Grand Total"}

for kolom in urutan_status + ["Grand Total"]:
    baris_grand_total[kolom] = tabel[kolom].sum()

tabel = pd.concat(
    [tabel, pd.DataFrame([baris_grand_total])],
    ignore_index=True
)

# Tampilkan hasil data
tabel

Masukkan tanggal (DD/MM/YYYY): 9/11/2026
Masukkan jam (HH.MM): 14.00


,No,Nama Toko,1. New,2. Packing,3. Pesanan Siap,4. Siap Kirim,5. Dalam Pengiriman,6. Tunda,7. Selesai,8. Batal,Grand Total
0,1,UF32 (DS DHARMAHUSADA)-(UZ01),40,0,17,35,0,0,1,2,95
1,2,2DV4 (DS BATAM CITY)-(2DZ1),6,1,5,22,51,0,11,1,97
2,3,2DV1 (DS NAGOYA BATAM)-(2DZ1),2,2,9,24,48,0,12,0,97
3,4,TI85 (DS BINTARO)-(KZ01),5,13,5,19,4,0,32,0,78
4,5,BJ49 (DS DC BANDUNG)-(BZ01),2,1,2,7,11,1,38,0,62
...,...,...,...,...,...,...,...,...,...,...,...
61,62,R860 (DS ABDESIR MKS)-(RZ01),0,0,3,5,21,0,106,0,135
62,63,R876 (DS HASANUDDIN GOWA)-(RZ01),3,3,8,17,7,0,117,0,155
63,64,UF29 (DS TENGGILIS SBY)-(UZ01),1,1,4,12,18,1,123,1,161
64,65,R899 (DS UJUNG PANDANG)-(RZ01),34,8,16,9,33,1,154,1,256


**BUAT EXCEL**

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from google.colab import files

# =====================================================
# BUAT WORKBOOK
# =====================================================

wb = Workbook()

ws = wb.active

ws.title = "UPDATE APO DARKSTORE"

# Hilangkan gridline
ws.sheet_view.showGridLines = False


# =====================================================
# JUDUL
# Mulai dari kolom B karena A sengaja kosong
# =====================================================

ws.merge_cells("B1:L1")

ws["B1"] = "UPDATE APO DARKSTORE"

ws["B1"].font = Font(
    bold=True,
    size=14
)

ws["B1"].alignment = Alignment(
    horizontal="left",
    vertical="center"
)


ws.merge_cells("B2:L2")

ws["B2"] = f"{tanggal} JAM {jam}"

ws["B2"].font = Font(
    bold=True,
    size=11
)

ws["B2"].alignment = Alignment(
    horizontal="left",
    vertical="center"
)


# =====================================================
# BORDER
# =====================================================

garis = Side(
    style="thin",
    color="000000"
)

border = Border(
    left=garis,
    right=garis,
    top=garis,
    bottom=garis
)


# =====================================================
# HEADER
# B:L
# =====================================================

for kolom, nama in enumerate(
    tabel.columns,
    2
):

    cell = ws.cell(
        row=4,
        column=kolom,
        value=nama
    )

    cell.font = Font(
        bold=True
    )

    cell.fill = PatternFill(
        fill_type="solid",
        fgColor="B7DEE8"
    )

    cell.alignment = Alignment(
        horizontal="center",
        vertical="center"
    )

    cell.border = border


# =====================================================
# ISI DATA
# B:L
# =====================================================

for baris in range(
    len(tabel)
):

    excel_baris = baris + 5

    for kolom in range(
        len(tabel.columns)
    ):

        nilai = tabel.iloc[
            baris,
            kolom
        ]

        cell = ws.cell(
            row=excel_baris,
            column=kolom + 2
        )

        cell.value = nilai

        cell.border = border

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center"
        )

        # Nama toko rata kiri
        if kolom == 1:

            cell.alignment = Alignment(
                horizontal="left",
                vertical="center"
            )


# =====================================================
# WARNA KOLOM 7. SELESAI
# MERAH → KUNING → HIJAU
# =====================================================

jumlah_toko = len(tabel) - 1

for i in range(jumlah_toko):

    if jumlah_toko > 1:
        posisi = i / (jumlah_toko - 1)
    else:
        posisi = 0

    # Merah → Kuning
    if posisi <= 0.5:

        persen = posisi / 0.5

        merah = 255
        hijau = int(255 * persen)

    # Kuning → Hijau
    else:

        persen = (posisi - 0.5) / 0.5

        merah = int(255 * (1 - persen))
        hijau = 255

    warna = (
        f"{merah:02X}"
        f"{hijau:02X}"
        "00"
    )

    # KOLOM J = 7. SELESAI
    cell = ws.cell(
        row=i + 5,
        column=10
    )

    cell.fill = PatternFill(
        fill_type="solid",
        fgColor=warna
    )

# =====================================================
# GRAND TOTAL
# MERGE B:C
# (No + Nama Toko)
# =====================================================

baris_total = len(tabel) + 4


ws.merge_cells(
    start_row=baris_total,
    start_column=2,
    end_row=baris_total,
    end_column=3
)


ws.cell(
    row=baris_total,
    column=2
).value = "Grand Total"


# =====================================================
# FORMAT GRAND TOTAL
# =====================================================

for kolom in range(
    2,
    13
):

    cell = ws.cell(
        row=baris_total,
        column=kolom
    )

    cell.fill = PatternFill(
        fill_type="solid",
        fgColor="B7DEE8"
    )

    cell.font = Font(
        bold=True
    )

    cell.alignment = Alignment(
        horizontal="center",
        vertical="center"
    )

    cell.border = border


# =====================================================
# LEBAR KOLOM
# =====================================================

# A = kolom kosong
ws.column_dimensions["A"].width = 3

# B:L = tabel
lebar = [
    7,   # B - No
    34,  # C - Nama Toko
    10,  # D - 1. New
    12,  # E - 2. Packing
    15,  # F - 3. Pesanan Siap
    13,  # G - 4. Siap Kirim
    20,  # H - 5. Dalam Pengiriman
    10,  # I - 6. Tunda
    11,  # J - 7. Selesai
    10,  # K - 8. Batal
    13   # L - Grand Total
]

for kolom, ukuran in enumerate(
    lebar,
    2
):

    ws.column_dimensions[
        get_column_letter(kolom)
    ].width = ukuran


# =====================================================
# TINGGI BARIS
# =====================================================

ws.row_dimensions[1].height = 22
ws.row_dimensions[2].height = 20
ws.row_dimensions[4].height = 30


# =====================================================
# FREEZE HEADER
# =====================================================

ws.freeze_panes = "B5"

# =====================================================
# LEBAR KOLOM
# =====================================================

# Kolom A = 63 pixel
ws.column_dimensions["A"].width = 8.4

# B:L = tabel
lebar = [
    7,   # B - No
    34,  # C - Nama Toko
    10,  # D - 1. New
    12,  # E - 2. Packing
    15,  # F - 3. Pesanan Siap
    13,  # G - 4. Siap Kirim
    20,  # H - 5. Dalam Pengiriman
    10,  # I - 6. Tunda
    11,  # J - 7. Selesai
    10,  # K - 8. Batal
    13   # L - Grand Total
]

for kolom, ukuran in enumerate(lebar, 2):
    ws.column_dimensions[
        get_column_letter(kolom)
    ].width = ukuran

# Kolom M = 63 pixel
ws.column_dimensions["M"].width = 8.4

# =====================================================
# SIMPAN
# =====================================================

nama_file = "UPDATE_APO_DARKSTORE.xlsx"

wb.save(
    nama_file
)

print(
    f"File berhasil dibuat: {nama_file}"
)

File berhasil dibuat: UPDATE_APO_DARKSTORE.xlsx


In [ ]:
files.download(
    "UPDATE_APO_DARKSTORE.xlsx"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>